# Overview:

While looking for web based UIs to support LangChain I came accross LangFlow.

For more information, consult LangFlow's [github page](https://github.com/langflow-ai/langflow) and the [official documentation](https://docs.langflow.org/).

# Commercial Support And Licensing

LangFlow was originally created by the startup named Logspace. On April 4th of 2024, DataStax [announced](https://www.datastax.com/press-release/datastax-acquires-langflow-to-make-building-genai-apps-100x-easier-faster-more-fun) it had entered into an agreement to acquire Logspace.

Currently LangFlow is provided under the MIT License.

# Architecture

## REST API

After looking at the source code, we can [see](https://github.com/langflow-ai/langflow/blob/dev/src/backend/base/langflow/server.py) that Gunicorn is being used to serve the webpage. Gunicorn 'Green Unicorn' is a Python WSGI HTTP Server for UNIX. If we continue digging, we [see](https://github.com/langflow-ai/langflow/blob/dev/src/backend/base/langflow/main.py) the underlying langflow API is implemented using the FastAPI library. We can see the routes defined [here](https://github.com/langflow-ai/langflow/blob/09fd88a0d06c7d0eb0b7f8b4ad991408f31729f8/src/backend/base/langflow/api/v1/endpoints.py).

### Async API

The documentation also [mentions](https://docs.langflow.org/guides/async-tasks) the Asynchronous API as being a feature which is currently still in development. This feature allows API calls to be performed asynchronously and requires an additional set of infrastructure. Specifically a Celery worker queue, Redis cache and a RabbitMQ message broker. This infrastructure provides the asynchronous functionality and corresponding API endpoints.

## Frontend

From looking at the source code, we can [see](https://github.com/langflow-ai/langflow/tree/09fd88a0d06c7d0eb0b7f8b4ad991408f31729f8/src/frontend) that the langflow frontend is implemented with ReactJS.

## VCS Integration

There is no direct VCS integration. That being said, the tool does allow for the import and export of Flows and other various configurations as JSON files. This will allow a human to manually execute a process to retrieve and store the workflows in the VCS tool of choice.

## RBAC

According to the [documentation](https://docs.langflow.org/guidelines/login), the tool does give you the ability to create an manage user accounts. 

> The login functionality in Langflow serves to authenticate users and protect sensitive routes in the application. Starting from version 0.5, Langflow introduces an enhanced login mechanism that is governed by a few environment variables. This allows new secure features.

There are two basic profiles which a user can assume: superuser and non-superuser. It appears that any flow created by a user are not visible to other users by default. An API key can be generated for a Flow once it has been setup, and this key will allow other users to leverage the Flow.

## Supported Frameworks

The homepage describes Langflow as:

> LangFlow is a GUI for LangChain, designed with react-flow to provide an effortless way to experiment and prototype flows with drag-and-drop components

And the following datastax blog states:

> Langflow is a new way of building AI apps built on top of LangChain that integrates with the full breadth of the new AI programming stack.

Additionally, from looking at the CustomComponent offered by the UI, it appears the user is able to define their own components using their own javascript libraries.

# Major Concepts

## Flows

A Flow is LangFlow's abstraction for an AI workflow which is expected to provide some sort of chat bot functionality. A Flow is implemented as a Directed Asyclic Graph (DAG) which defines the flow of data between Components.

Looking at the UI, there are several ways to create a Flow. I believe the term Flow and Project refer to the same logical construct. Looking at the UI, creating a Flow or a Project lead to the same screen. 

<center><img src="images\langflow_create_flow.png"></center>

After clicking either button, we are lead to a page with a Component menu and canvas allowing users to drag and drop components and then define the links between them.

<center><img src="images\langflow_create_flow_2.png"></center>

## Components

The Components provide the ability for a Flow to leverage functionality from external providers. Each Component abstracts the respective logic and configurations required to integrate and leverage a third party API. From the Flow Canvas's Component Menu, we can see that there is a large list of Components providing off-the-shelf integrations. 

After looking at the [LangFlow documentation](https://docs.langflow.org/getting-started/creating-flows), it appears that the Componets are wrappers around the LangChain framework. The [LangChain documentation](https://python.langchain.com/docs/modules/) in turn also uses the term "Components" to refer to the set of standard extendable interfaces and external integrations provided by the framework.

Looking at the Menu, we can see that compoents are logically sorted into categories. For example, when looking at the LLMs category, we see that a Flow is able to connect to a number of different LLM providers.

<center><img src="images\langflow_components.png"></center>

Currently LangFlow offers the following categories of Components:

- Custom
- Agents
- Chains
- Loaders
- Embeddings
- LLMs
- Memories
- Output Parsers
- Prompts
- Retrievers
- Text Splitters
- Toolkits
- Tools
- Utilities
- Vector Stores
- Wrappers

### Assistants & Agents

Generally speaking the concept of Assistant and Agents are used interchangeably. An Assistant or Agent is a wrapper around an LLM. The agent proxies requests from the user. In some cases, the agent will administer a prompt establishing the specific function and/or desired response format from the LLM.

Major LLM providers like OpenAI and IBM Watson offer Assistant APIs for their LLM offerings. But it is also possible for open source off-the-shelf agents to exist for the open-source LLM offerings.

The way to leverage an Assistant using FlowiseAI is through a Chatflow; specifically through the Nodes in the chatflow. FlowiseAI offers a number of Nodes providing Agent functionality which in turn wraps around an LLM. In the case of a vendor providing an assistant API, the FlowiseAI Agent is typically just a wrapper around the vendor's agent API.

This is the case when using OpenAI. When we create a new OpenAI Agent, FlowiseAI will perform the backend API calls to properly configure the Agent within OpenAI's platform. The FlowiseAI platform will then handle the logic of redirecting the user input provided to the chat bot to the OpenAI Assistant and then relaying the response back to the UI.

### Chains

While Chains have a framework specific implementation, they can be generally defined as being a sequential set of operations that a framework is used to perform, which may be interdependent.

Confusingly, while a chain conceptually represents a collection of actions, the Components in LangFlow represent a single action. Thus if we wanted to have multiple actions chained together, we would need to define multiple Components and connect them accordingly. 

### Memory
This group of Components provides the ability to store chat history and other information to storage mediums and then access that information as part of the Flow. The mediums include MongoDB, Motorhead, and more.

### Output Parsers
These Components are used to perform transformations on the LLM outputs.

### Prompts
These Components allow the user to specify a prompt for a given LLM interaction. They can be used to custom define chat assistants.

### Vector Stores

These Components provide integrations with propular Vector Stores allowing the Flow to access the vectors stored in the Vector Store.


# Installation

**Note**: Hugging face offers a cloud based service through HuggingFace Spaces.

According to the official installation instructions, the LangFlow solution is packaged as a python module and can be installed via pip. The current latest version is 0.6.15. Below ware the commands I used to install this package inside a conda environment:

```
(base) [root@fedora ~]# conda create -n langflow-3.10 python=3.10
(base) [root@fedora ~]# conda activate langflow-3.10
(langflow-3.10) [root@fedora ~]# python -m pip install langflow==0.6.15
```

This installed the latest stable release 

```
(langflow-3.10) [root@fedora ~]# pip list | grep langflow
langflow                                 0.6.15
```

You can run the application with a special instruction so that it binded to all IPs
```
(langflow-3.10) [root@fedora ~]# langflow run --host 0.0.0.0
```

There appears to be a security feature in some browsers which prevents this application from running when accessed via an external IP (i.e. not 127.0.0.1). This is documented in the [PR](https://github.com/langflow-ai/langflow/issues/1692) I opened. The current workaround is as follows:

> 1. Open Chrome and navigate to chrome://flags in the address bar.
> 2. Search for the flag titled Insecure origins treated as secure.
> 3. In the text box for this flag, enter the URL of your site.
> 4. Enable this setting.
> 5. Restart Chrome.

After performing this workaround, the solution will properly load.